In [28]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt
import torchvision.models as models

In [39]:
from pathlib import Path

data_dir = Path("/workspaces/DSC140B-CODING-HW/data/data")
image_paths = sorted(data_dir.glob("*.jpg"))

print("Number of images:", len(image_paths))
print(image_paths[:5])

Number of images: 9181
[PosixPath('/workspaces/DSC140B-CODING-HW/data/data/Anaheim-0021450d.jpg'), PosixPath('/workspaces/DSC140B-CODING-HW/data/data/Anaheim-00380032.jpg'), PosixPath('/workspaces/DSC140B-CODING-HW/data/data/Anaheim-00752b22.jpg'), PosixPath('/workspaces/DSC140B-CODING-HW/data/data/Anaheim-011a4d5d.jpg'), PosixPath('/workspaces/DSC140B-CODING-HW/data/data/Anaheim-01704105.jpg')]


In [46]:
labels_str = [p.stem.split("-")[0] for p in image_paths]

print(labels_str[:10])
print(sorted(set(labels_str)))

['Anaheim', 'Anaheim', 'Anaheim', 'Anaheim', 'Anaheim', 'Anaheim', 'Anaheim', 'Anaheim', 'Anaheim', 'Anaheim']
['Anaheim', 'Bakersfield', 'Los_Angeles', 'Riverside', 'SLO', 'San_Diego']


## RestNet18

In [40]:
# load ResNet-18
ft_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

In [ ]:
# freeze everything
for param in ft_model.parameters():
    param.requires_grad = False

# unfreeze the last convolutional block for learning features
for param in ft_model.layer4.parameters():
    param.requires_grad = True

In [42]:
# execuation location
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [53]:
# replace the final layer with 6-class classifier for proj requirement. Quit the frozen final layer directly
ft_model.fc = nn.Linear(512, 6)
ft_model = ft_model.to(device)

In [54]:
# choose the loss function
ft_loss_fn = nn.CrossEntropyLoss()

# choose optimizer
ft_optimizer = torch.optim.Adam([
    {'params': ft_model.layer4.parameters(), 'lr': 1e-4},
    {'params': ft_model.fc.parameters(),     'lr': 1e-3},
])

In [55]:
# split the data
from sklearn.model_selection import train_test_split

train_paths, val_paths = train_test_split(
    image_paths,
    test_size=0.2,
    random_state=42,
    stratify=labels_str
)

print(len(train_paths), len(val_paths))

7344 1837


In [56]:
# build label mapping
classes = sorted(set(labels_str))
class_to_idx = {c: i for i, c in enumerate(classes)}

print(classes)
print(class_to_idx)

['Anaheim', 'Bakersfield', 'Los_Angeles', 'Riverside', 'SLO', 'San_Diego']
{'Anaheim': 0, 'Bakersfield': 1, 'Los_Angeles': 2, 'Riverside': 3, 'SLO': 4, 'San_Diego': 5}


In [ ]:
# define transforms
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [50]:
# define the dataset
from PIL import Image
from torch.utils.data import Dataset

class SoCalDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        image = Image.open(path).convert("RGB")

        label_str = path.stem.split("-")[0]
        label = class_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label

In [36]:
def evaluate(model, loader):
    model.eval()
    correct, n = 0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            pred = model(X_batch)
            correct += (pred.argmax(dim=1) == y_batch).sum().item()
            n += len(y_batch)

    return correct / n

In [51]:
# create train_loader and val_loader
from torch.utils.data import DataLoader

train_dataset = SoCalDataset(train_paths, transform=train_transform)
val_dataset = SoCalDataset(val_paths, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

X, y = next(iter(train_loader))
print(X.shape)
print(y.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [52]:
n_batches = len(train_loader)

best_val_acc = 0
best_state = None

for epoch in range(8):
    # --- training ---
    ft_model.train()
    total_loss, correct, n = 0.0, 0, 0

    for batch_idx, (X_batch, y_batch) in enumerate(train_loader, 1):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        pred = ft_model(X_batch)
        loss = ft_loss_fn(pred, y_batch)

        ft_optimizer.zero_grad()
        loss.backward()
        ft_optimizer.step()

        total_loss += loss.item() * len(y_batch)
        correct += (pred.argmax(dim=1) == y_batch).sum().item()
        n += len(y_batch)

        print(f"\r  Epoch {epoch+1:2d}/8  "
              f"batch {batch_idx}/{n_batches}  "
              f"loss={total_loss/n:.4f}  "
              f"train_acc={correct/n:.3f}", end="")

    # --- evaluation ---
    val_acc = evaluate(ft_model, val_loader)

    print(f"\r  Epoch {epoch+1:2d}/8  "
          f"loss={total_loss/n:.4f}  "
          f"train_acc={correct/n:.3f}  "
          f"val_acc={val_acc:.3f}      ")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = ft_model.state_dict()

  Epoch  1/8  loss=0.6493  train_acc=0.771  val_acc=0.870      
  Epoch  2/8  loss=0.3346  train_acc=0.884  val_acc=0.886      
  Epoch  3/8  loss=0.2341  train_acc=0.920  val_acc=0.879      
  Epoch  4/8  loss=0.1631  train_acc=0.943  val_acc=0.901      
  Epoch  5/8  loss=0.1199  train_acc=0.961  val_acc=0.895      
  Epoch  6/8  loss=0.0879  train_acc=0.970  val_acc=0.896      
  Epoch  7/8  loss=0.0764  train_acc=0.975  val_acc=0.900      
  Epoch  8/8  loss=0.0830  train_acc=0.969  val_acc=0.901      
